# S6E9 — What Actually Moves AUC: An Honest Ablation

Most of what I tried on this competition **did nothing**. That turned out to be
the useful finding, so this notebook reports the failures alongside the wins.

Everything below is a 5-fold out-of-fold AUC on the full 668k training rows.

| change | OOF AUC | delta |
|---|---|---|
| LightGBM, raw features | 0.94167 | — |
| \+ original source dataset merged | 0.94170 | **+0.00003** |
| \+ hand-built ratio / interaction features | 0.94169 | **−0.00002** |
| \+ heavy hyperparameter tuning | 0.94166 | **−0.00001** |
| XGBoost, raw features | 0.94193 | — |
| **\+ digit features** | 0.94353 | **+0.0016** |
| **\+ frequency encoding** | 0.94411 | **+0.0006** |
| final LGB+XGB+Cat blend | **0.94440** | |

**The takeaway:** this data is *synthetic* — generated by a model fitted to a
real dataset. A plain GBDT already extracts essentially all of the **semantic**
signal, which is why domain-flavoured feature engineering and tuning buy you
nothing. What pays is exposing the **generator's fingerprints**.

Two transforms do almost all the work:

1. **Digit features** — the decimal expansion of numeric columns (leading and
   trailing digits, digit-length, fractional part). The generator's sampling
   shows up in digit positions that the raw value hides.
2. **Frequency encoding** — how often each value occurs, computed over
   **train + test together**. Splitting it destroys the signal.

Credit where due: the digit + frequency combination is the standard Playground
toolkit, and the clearest public write-up of it for S6E9 is
[evgendvorkin's Single XGB notebook](https://www.kaggle.com/code/evgendvorkin/s6e9-single-xgb-cv-0-94583).
The implementation here is my own; the ablation numbers and the negative
results are mine.

For calibration: the best public CV I have seen on S6E9 is **0.94583**, so the
0.94440 here does not close the whole gap — there is still something in that
notebook I have not reproduced. I would rather say so than quietly imply this
is the ceiling.


## Setup


In [ ]:
import warnings, time
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# QUICK = True sanity-checks the plumbing in ~2 minutes on 60k rows.
#
# WARNING: do not read the ablation off a QUICK run. Digit + frequency encoding
# add ~150 features, and on a 60k subsample with shallow trees they measure as
# *negative* (-0.0002) because there are not enough rows to fit them. The gains
# below are real only on the full 668k. This bit me, so it is flagged here.
QUICK = False
FOLDS = 5

# Locate the competition data wherever Kaggle mounted it. The mount point has
# moved between /kaggle/input/<comp>/ and /kaggle/input/competitions/<comp>/,
# so glob for it -- this also makes the notebook portable when you fork it for
# the next season.
import glob, os
hits = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True))
DATA = os.path.dirname(hits[0]) if hits else '.'
print('data dir:', DATA)

train = pd.read_csv(f'{DATA}/train.csv')
test  = pd.read_csv(f'{DATA}/test.csv')
if QUICK:
    train = train.sample(60000, random_state=0).reset_index(drop=True)

TARGET = 'Will_Buy_EV'
y = (train[TARGET] == 'Yes').astype(int).values
FEAT = [c for c in train.columns if c not in ('id', TARGET)]
NUM  = [c for c in FEAT if train[c].dtype.kind in 'if']
CAT  = [c for c in FEAT if c not in NUM]
print(f'train {train.shape}  test {test.shape}  positives {y.mean():.4f}')
print(f'{len(NUM)} numeric, {len(CAT)} categorical')


## 1. Rule out the cheap wins first

Before modelling anything, three things are worth 30 seconds because when they
*do* exist they dominate everything else: duplicate feature-rows with a
deterministic label, train/test row overlap, and signal in `id`.

All three are absent here — but they are present in some Playground seasons, so
the check is worth keeping in your template.


In [ ]:
key_tr = train[FEAT].astype(str).agg('|'.join, axis=1)
key_te = test[FEAT].astype(str).agg('|'.join, axis=1)

n_dupe = len(key_tr) - key_tr.nunique()
overlap = key_te.isin(set(key_tr)).mean()
id_corr = np.corrcoef(train['id'], y)[0, 1]

print(f'duplicate feature-rows in train : {n_dupe}')
print(f'test rows also seen in train    : {overlap*100:.2f}%')
print(f'corr(id, target)                : {id_corr:.5f}')
print('target rate by id decile        :',
      [round(v, 4) for v in pd.Series(y).groupby(pd.qcut(train["id"], 10, labels=False)).mean()])


## 2. The two transforms that matter

`digit_features` pulls apart the decimal expansion of every numeric column.
`frequency_encode` adds normalised value counts — computed on train and test
**concatenated**, which is the part people get wrong.


In [ ]:
def digit_features(df, cols):
    """Expose the decimal expansion of numeric columns."""
    out = df.copy()
    for c in cols:
        v = out[c].astype('float64').fillna(0)
        s = (v * 10).round().astype('int64').abs().astype(str)
        rev = s.str[::-1]
        out[f'{c}_len']  = s.str.len().astype('int8')
        out[f'{c}_frac'] = ((v - np.floor(v)) * 100).round().astype('int16')
        for k in range(4):
            out[f'{c}_d{k}'] = s.str[k:k+1].replace('', '-1').astype('int8')
            out[f'{c}_e{k}'] = rev.str[k:k+1].replace('', '-1').astype('int8')
    return out


def frequency_encode(df, cols=None):
    """Normalised occurrence rate of each value, over the whole dataset."""
    out = df.copy()
    for c in (cols if cols is not None else list(df.columns)):
        out[f'{c}_fq'] = out[c].map(out[c].value_counts(normalize=True)).astype('float32')
    return out


def make_features(train, test, feat, digits=True, freq=True):
    n = len(train)
    num = [c for c in feat if train[c].dtype.kind in 'if']
    cat = [c for c in feat if c not in num]
    both = pd.concat([train[feat], test[feat]], ignore_index=True)
    for c in cat:
        codes = {v: i for i, v in enumerate(both[c].astype(str).unique())}
        both[c] = both[c].astype(str).map(codes).astype('int32')
    if digits and num:
        both = digit_features(both, num)
    if freq:
        both = frequency_encode(both)
    return both.iloc[:n].reset_index(drop=True), both.iloc[n:].reset_index(drop=True)


X_raw, T_raw = make_features(train, test, FEAT, digits=False, freq=False)
X,     T     = make_features(train, test, FEAT, digits=True,  freq=True)
print(f'raw: {X_raw.shape[1]} features   ->   engineered: {X.shape[1]} features')


## 3. The ablation

Same model, same folds, same seed — only the feature set changes.


In [ ]:
def cv_auc(X, label, folds=FOLDS, seed=0):
    oof = np.zeros(len(X)); t0 = time.time()
    for a, b in StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y):
        m = xgb.XGBClassifier(
            learning_rate=0.03, max_depth=8, subsample=0.85, colsample_bytree=0.5,
            n_estimators=4000, early_stopping_rounds=150, tree_method='hist',
            eval_metric='auc', min_child_weight=20, reg_lambda=2.0,
            random_state=seed, verbosity=0)
        m.fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])], verbose=False)
        oof[b] = m.predict_proba(X.iloc[b])[:, 1]
    auc = roc_auc_score(y, oof)
    print(f'{label:34s} {auc:.5f}   ({X.shape[1]:3d} feats, {time.time()-t0:.0f}s)')
    return auc, oof

a_raw,  _ = cv_auc(X_raw, 'raw features')
X_dig, _T = make_features(train, test, FEAT, digits=True, freq=False)
a_dig,  _ = cv_auc(X_dig, '+ digit features')
a_all, oof_xgb = cv_auc(X, '+ digit + frequency encoding')

print(f'\ndigit features    : {a_dig - a_raw:+.5f}')
print(f'frequency encoding: {a_all - a_dig:+.5f}')


## 4. Final model — three GBDTs, blended on OOF

Nothing exotic: LightGBM, XGBoost and CatBoost on the same feature set, with a
greedy hill-climb blend fitted on out-of-fold predictions.


In [ ]:
oof = {m: np.zeros(len(X)) for m in ('lgb', 'xgb', 'cat')}
pred = {m: np.zeros(len(T)) for m in ('lgb', 'xgb', 'cat')}

for k, (a, b) in enumerate(StratifiedKFold(FOLDS, shuffle=True, random_state=0).split(X, y)):
    t0 = time.time()
    m = lgb.LGBMClassifier(objective='binary', learning_rate=0.03, num_leaves=96,
        n_estimators=5000, feature_fraction=0.5, bagging_fraction=0.85, bagging_freq=1,
        min_child_samples=60, lambda_l2=2.0, verbosity=-1, seed=0)
    m.fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])],
          callbacks=[lgb.early_stopping(150, verbose=False)])
    oof['lgb'][b] = m.predict_proba(X.iloc[b])[:, 1]
    pred['lgb'] += m.predict_proba(T)[:, 1] / FOLDS

    m = xgb.XGBClassifier(learning_rate=0.03, max_depth=8, subsample=0.85,
        colsample_bytree=0.5, n_estimators=5000, early_stopping_rounds=150,
        tree_method='hist', eval_metric='auc', min_child_weight=20,
        reg_lambda=2.0, random_state=0, verbosity=0)
    m.fit(X.iloc[a], y[a], eval_set=[(X.iloc[b], y[b])], verbose=False)
    oof['xgb'][b] = m.predict_proba(X.iloc[b])[:, 1]
    pred['xgb'] += m.predict_proba(T)[:, 1] / FOLDS

    m = CatBoostClassifier(learning_rate=0.06, depth=8, iterations=5000, verbose=0,
        early_stopping_rounds=150, random_seed=0, eval_metric='AUC')
    m.fit(X.iloc[a], y[a], eval_set=(X.iloc[b], y[b]))
    oof['cat'][b] = m.predict_proba(X.iloc[b])[:, 1]
    pred['cat'] += m.predict_proba(T)[:, 1] / FOLDS
    print(f'fold {k}  {time.time()-t0:.0f}s')

for m in oof:
    print(f'{m}: {roc_auc_score(y, oof[m]):.5f}')


In [ ]:
def hill_climb(oof, y, steps=60):
    names = list(oof); w = {n: 0.0 for n in names}
    cur = np.zeros_like(oof[names[0]]); best = -np.inf
    for _ in range(steps):
        gain, pick = None, None
        for n in names:
            cand = (cur * sum(w.values()) + oof[n]) / (sum(w.values()) + 1)
            s = roc_auc_score(y, cand)
            if gain is None or s > gain:
                gain, pick = s, n
        if gain <= best + 1e-9:
            break
        best = gain
        cur = (cur * sum(w.values()) + oof[pick]) / (sum(w.values()) + 1)
        w[pick] += 1
    tot = sum(w.values()) or 1
    return {n: v / tot for n, v in w.items()}, best

w, blend_auc = hill_climb(oof, y)
print('blend weights   :', {k: round(v, 3) for k, v in w.items()})
print(f'blended OOF AUC : {blend_auc:.5f}')

blend = sum(w[m] * pred[m] for m in pred)
sub = pd.DataFrame({'id': test['id'], TARGET: blend})
sub.to_csv('submission.csv', index=False)
sub.head()


## What I'd take to the next Playground

0. **More features need more rows.** On the full 668k these 150 extra columns are
   worth +0.0022; on a 60k subsample the same columns measure −0.0002. If you
   fork this and evaluate on a sample, you will conclude the opposite of the
   truth.
1. **Check for duplicate rows, train/test overlap and `id` signal first.** Thirty
   seconds, and when one of them exists it beats every model you could build.
2. **Don't bother with domain feature engineering on synthetic data.** Ratios and
   interactions that "should" help measured at −0.00002 here.
3. **Don't over-tune.** Heavy hyperparameter search came in at −0.00001 against
   sensible defaults.
4. **Do go after generator fingerprints** — digit features and frequency
   encoding were worth ~+0.0022 combined, i.e. roughly *a hundred times*
   everything else put together.
5. **Finding the original source dataset is not automatically worth it.** The
   source here ([EV Adoption Behavior and Range Anxiety](https://www.kaggle.com/datasets/itzzomkar/ev-adoption-behavior-and-range-anxiety),
   an exact column match) is 10k rows against 668k competition rows, so it moved
   +0.00003. Only worth merging when the original is large relative to the
   competition data.

Fork it, change `DATA` and `TARGET`, and the whole thing runs on the next
season's data.

*If the negative results saved you an afternoon, an upvote is appreciated.*
